# Domain 3: Claude Code Configuration & Workflows
## TS 3.1: Configure CLAUDE.md Files with Appropriate Hierarchy and Scoping
**Certification Focus**: CCAR-F Domain 3 (20% Exam Weight) | **Scenario Alignment**: Scenario 2 & 5

---

### 🎯 Key Architectural Principles
1. **The 5-Layer Configuration Hierarchy**:
   - Layer 1: **Enterprise Managed Policies** (`managed-policy.json`) - immutable constraints set by system administrators.
   - Layer 2: **User Global Settings** (`~/.claude/CLAUDE.md`) - user-level preferences across all repositories.
   - Layer 3: **Project Root Configuration** (`.claude/CLAUDE.md` or repo `CLAUDE.md`) - repository-wide standards.
   - Layer 4: **Path-Scoped Rules** (`.claude/rules/*.md`) - dynamically loaded conventions based on edited file paths.
   - Layer 5: **Local Invocation Flags** (CLI arguments).
2. **Precedence Contract**: More specific project and path rules extend or override general user rules, but cannot violate enterprise managed policies.
3. **Headless CI Review**: Invoking Claude Code in CI/CD requires `-p` / `--non-interactive` mode and deterministic exit codes.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.1 Claude Code Workflows.")

### 📜 Step 2: Path-Scoped Rules Engine & CI Review Simulator
We define the rule store and implement rule matching and CI review execution.

In [ ]:
# Step 2: Rule Matching & CI Simulator
@dataclass
class PathScopedRule:
    rule_name: str
    globs: List[str]
    rule_content: str

    def matches(self, filepath: str) -> bool:
        return any(fnmatch.fnmatch(filepath, pattern) for pattern in self.globs)

RULE_STORE: List[PathScopedRule] = [
    PathScopedRule(
        rule_name="api_conventions",
        globs=["src/api/**/*", "**/api/**/*.py"],
        rule_content="All API handlers must use explicit Pydantic response models and async/await.",
    ),
    PathScopedRule(
        rule_name="test_conventions",
        globs=["**/*.test.*", "tests/**/*.py"],
        rule_content="All tests must use pytest patterns and include docstrings with AAA structure.",
    ),
]

def resolve_active_rules_for_file(filepath: str) -> List[str]:
    """Determines which rules apply to a given file path."""
    active = []
    for rule in RULE_STORE:
        if rule.matches(filepath):
            active.append(rule.rule_content)
    return active

def simulate_ci_review_execution(git_diff: str, active_rules: List[str], is_non_interactive: bool = True) -> Dict[str, Any]:
    """Simulates headless CI audit (-p mode). Fails if interactive input is expected."""
    if not is_non_interactive:
        raise RuntimeError("Missing -p flag: CI/CD runners must execute in non-interactive mode!")
    
    findings = []
    if "eval(" in git_diff or "exec(" in git_diff:
        findings.append({
            "severity": "CRITICAL",
            "rule": "SEC001_NO_DYNAMIC_EXECUTION",
            "message": "Dynamic code execution (eval/exec) strictly prohibited in pull requests."
        })
    return {
        "status": "completed",
        "total_findings": len(findings),
        "findings": findings,
        "exit_code": 1 if findings else 0,
    }

print(" Rule matcher and CI reviewer initialized.")

### 🧪 Step 3: Interactive Simulation
1. Test rule resolution for test vs API files.
2. Simulate CI review of a PR containing dangerous `eval()` code.

In [ ]:
# Step 3: Simulation
# 1. Rule Resolution
test_rules = resolve_active_rules_for_file("tests/unit/test_auth.py")
print("Active Rules for test_auth.py:", test_rules)

api_rules = resolve_active_rules_for_file("src/api/v1/orders.py")
print("Active Rules for orders.py:", api_rules)

# 2. CI/CD Audit Simulation
unsafe_diff = """
@@ -10,3 +10,4 @@
 def handle_request(raw_code):
+    return eval(raw_code)
"""
ci_result = simulate_ci_review_execution(unsafe_diff, active_rules=api_rules, is_non_interactive=True)
print("\nCI Review Result:", json.dumps(ci_result, indent=2))

### 🔬 Step 4: Verification Assertions
We assert that:
1. Test rules apply only to test paths.
2. API rules apply only to API paths.
3. CI review correctly flags `eval()` as critical with exit code 1.
4. Non-interactive mode enforcement raises RuntimeError if flag is missing.

In [ ]:
# Step 4: Verification Checks
assert any("pytest" in r for r in test_rules)
assert any("Pydantic" in r for r in api_rules)

assert ci_result["exit_code"] == 1
assert ci_result["findings"][0]["severity"] == "CRITICAL"

# Verify non-interactive safeguard
try:
    simulate_ci_review_execution(unsafe_diff, api_rules, is_non_interactive=False)
    assert False, "Should have failed without non-interactive mode!"
except RuntimeError as exc:
    assert "Missing -p flag" in str(exc)

print(" All verification assertions passed! Configuration hierarchy verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Know the 5 hierarchy levels: Enterprise > User Global > Project Root > Path Rules > CLI Flags.
- **Rule 2**: Use path-specific rules to conditionally load conventions only when matching files are edited.
- **Rule 3**: In CI/CD pipelines, always pass `-p` / `--non-interactive` to prevent hanging processes.